# 🏦 Loan Amortization — Interactive Strategy Comparison

Compare three repayment strategies side-by-side. Drag the sliders and the entire analysis updates instantly.

| # | Strategy | How it works |
|---|----------|-------------|
| 1 | **Regular** | Pay only the EMI — no extra payments, full original tenure |
| 2 | **Prepayment** | Make a fixed annual lump-sum prepayment to close the loan in a target tenure |
| 3 | **OD + Prepayment** | Park money in an overdraft account (higher rate) AND make annual prepayments |

In [ ]:
import sys, os

# Local dev fallback: add src/ to path if the package is not installed
_src = os.path.join(os.path.dirname(os.path.abspath("__file__")), "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from loan.amortization import (
    calculate_emi,
    build_schedule,
    build_schedule_with_prepayments,
    build_schedule_with_overdraft,
    build_schedule_with_od_and_prepayments,
)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110, 'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa', 'axes.grid': True,
    'grid.alpha': 0.3, 'grid.linestyle': '--',
})
print('Ready')

In [6]:
# ── Helpers ───────────────────────────────────────────────
def fmt(val):
    if abs(val) >= 1e7:  return f'₹{val/1e7:,.2f} Cr'
    if abs(val) >= 1e5:  return f'₹{val/1e5:,.2f} L'
    return f'₹{val:,.0f}'

def find_annual_prepay(principal, rate, orig_months, target_months, builder, **kw):
    """Binary search for annual prepayment to hit target tenure."""
    zero = builder(principal, rate, orig_months, prepayments={}, **kw)
    if zero.tenure_months <= target_months:
        return 0.0, zero
    lo, hi = 0.0, float(principal)
    nyrs = target_months // 12
    for _ in range(120):
        mid = (lo + hi) / 2
        pp = {m * 12: mid for m in range(1, nyrs + 1)}
        s = builder(principal, rate, orig_months, prepayments=pp, **kw)
        if s.tenure_months > target_months: lo = mid
        else: hi = mid
    amt = round(hi, 0)
    pp = {m * 12: amt for m in range(1, nyrs + 1)}
    return amt, builder(principal, rate, orig_months, prepayments=pp, **kw)

def invest_corpus(annual_amount, years, annual_return_pct):
    """Future value of investing annual_amount each year for N years (SIP-style)."""
    if annual_return_pct == 0 or annual_amount == 0 or years == 0:
        return annual_amount * years
    r = annual_return_pct / 100
    return annual_amount * ((1 + r) ** years - 1) / r

def lumpsum_fv(principal, years, annual_return_pct):
    """Future value of a lump-sum investment."""
    if annual_return_pct == 0 or principal == 0 or years == 0:
        return principal
    return principal * (1 + annual_return_pct / 100) ** years


# ── Widgets ──────────────────────────────────────────────
S = {'description_width': '160px'}
L = widgets.Layout(width='380px')

w_loan    = widgets.FloatSlider(value=150, min=10, max=500, step=5,
              description='Loan Amount (₹ Lakhs):', style=S, layout=L, readout_format='.0f',
              continuous_update=False)
w_rate    = widgets.FloatSlider(value=7.25, min=4.0, max=15.0, step=0.05,
              description='Interest Rate (%):', style=S, layout=L, readout_format='.2f',
              continuous_update=False)
w_tenure  = widgets.IntSlider(value=20, min=5, max=30, step=1,
              description='Original Tenure (yrs):', style=S, layout=L,
              continuous_update=False)
w_target  = widgets.IntSlider(value=10, min=3, max=29, step=1,
              description='Target Tenure (yrs):', style=S, layout=L,
              continuous_update=False)
w_od_rate = widgets.FloatSlider(value=7.60, min=4.0, max=15.0, step=0.05,
              description='OD Interest Rate (%):', style=S, layout=L, readout_format='.2f',
              continuous_update=False)
w_od_bal  = widgets.FloatSlider(value=20, min=0, max=500, step=5,
              description='OD Parked Bal (₹ Lakhs):', style=S, layout=L, readout_format='.0f',
              continuous_update=False)
w_invest  = widgets.FloatSlider(value=12.0, min=0, max=25.0, step=0.5,
              description='Invest Return (% p.a.):', style=S, layout=L, readout_format='.1f',
              continuous_update=False)

out = widgets.Output()

# ── Core update logic ────────────────────────────────────
def refresh(_=None):
    # Guard: target must be < original
    if w_target.value >= w_tenure.value:
        w_target.value = w_tenure.value - 1
        return
    if w_od_bal.value > w_loan.value:
        w_od_bal.value = w_loan.value
        return

    with out:
        clear_output(wait=True)

        loan       = w_loan.value * 1e5
        rate       = w_rate.value
        orig_mo    = w_tenure.value * 12
        target_mo  = w_target.value * 12
        od_rate    = w_od_rate.value
        od_bal     = w_od_bal.value * 1e5

        # ── Strategy 1: Regular ──
        s1 = build_schedule(loan, rate, orig_mo)

        # ── Strategy 2: Prepayment only ──
        pp2, s2 = find_annual_prepay(
            loan, rate, orig_mo, target_mo,
            lambda p,r,m,prepayments: build_schedule_with_prepayments(p,r,m,prepayments))

        # ── Strategy 3: OD + Prepayment ──
        pp3, s3 = find_annual_prepay(
            loan, od_rate, orig_mo, target_mo,
            lambda p,r,m,prepayments,od_balance=od_bal: build_schedule_with_od_and_prepayments(p,r,m,od_balance,prepayments))

        saved2 = s1.total_interest - s2.total_interest
        saved3 = s1.total_interest - s3.total_interest

        # ── Strategy 4: OD only — same parked balance, no prepayments ──
        s4 = build_schedule_with_overdraft(loan, od_rate, orig_mo, od_bal)
        saved4 = s1.total_interest - s4.total_interest

        # ── Opportunity cost: what if you invested instead? ──
        inv_ret = w_invest.value
        tgt_yrs = w_target.value
        # Strategy 2: annual prepayment invested instead (SIP over tgt_yrs)
        corpus_pp2 = invest_corpus(pp2, tgt_yrs, inv_ret) if pp2 else 0
        # Strategy 3: annual prepayment invested (SIP over tgt_yrs)
        corpus_pp3 = invest_corpus(pp3, tgt_yrs, inv_ret) if pp3 else 0
        # Strategy 3: OD parked lump-sum — locked for actual S3 tenure
        s3_yrs = s3.tenure_months / 12
        corpus_od_lump = lumpsum_fv(od_bal, s3_yrs, inv_ret) if od_bal else 0
        # Total capital deployed per strategy
        capital_pp2 = pp2 * tgt_yrs
        capital_pp3 = pp3 * tgt_yrs + od_bal

        # ── Net cash flow: prepay benefit vs investment opportunity cost ──
        # Prepay: money is gone → you saved interest, but missed market gains
        # OD: parked balance returns to you → opportunity cost is only the GROWTH
        od_missed_growth = corpus_od_lump - od_bal if od_bal else 0
        net_pp2 = saved2 - corpus_pp2                   # +ve = prepay was better
        net_pp3 = saved3 - corpus_pp3 - od_missed_growth  # +ve = OD+prepay was better

        # ── HTML Summary Table ──
        def R(lbl, v1, v2, v3, v4, hl=False):
            b = 'font-weight:700;' if hl else ''
            return (f'<tr style="border-bottom:1px solid #333;">'
                    f'<td style="padding:9px 14px;{b}">{lbl}</td>'
                    f'<td style="text-align:center;padding:9px;">{v1}</td>'
                    f'<td style="text-align:center;padding:9px;background:rgba(255,255,255,0.03);">{v2}</td>'
                    f'<td style="text-align:center;padding:9px;">{v3}</td>'
                    f'<td style="text-align:center;padding:9px;background:rgba(255,255,255,0.03);">{v4}</td></tr>')

        def mf(m): return f'{m} mo ({m//12}y {m%12}m)'

        # Pre-compute styled values that need conditional logic
        _clr2 = '#69f0ae' if net_pp2 >= 0 else '#ef5350'
        _clr3 = '#69f0ae' if net_pp3 >= 0 else '#ef5350'
        _sign2 = '+' if net_pp2 >= 0 else ''
        _sign3 = '+' if net_pp3 >= 0 else ''
        _net2_html = f'<span style="color:{_clr2};font-weight:700">{_sign2}{fmt(net_pp2)}</span>'
        _net3_html = f'<span style="color:{_clr3};font-weight:700">{_sign3}{fmt(net_pp3)}</span>'
        _inv2_html = fmt(corpus_pp2) if corpus_pp2 else '—'
        _inv3_val = corpus_pp3 + corpus_od_lump
        _inv3_html = fmt(_inv3_val) if _inv3_val else '—'
        # OD-only opportunity cost — locked for actual S4 tenure
        s4_yrs = s4.tenure_months / 12
        _corpus_od4 = lumpsum_fv(od_bal, s4_yrs, inv_ret) if od_bal else 0
        _od4_missed = _corpus_od4 - od_bal if od_bal else 0
        _net_od4 = saved4 - _od4_missed
        _clr4 = '#69f0ae' if _net_od4 >= 0 else '#ef5350'
        _sign4 = '+' if _net_od4 >= 0 else ''
        _net4_html = f'<span style="color:{_clr4};font-weight:700">{_sign4}{fmt(_net_od4)}</span>'
        _inv4_html = fmt(_corpus_od4) if _corpus_od4 else '—'

        tbl = f'''
        <div style="background:#1a1a2e;border-radius:12px;padding:24px;color:#ccc;
                    font-family:-apple-system,BlinkMacSystemFont,sans-serif;font-size:13px;">
        <h2 style="margin:0 0 16px 0;color:#fff;">⚡ Strategy Comparison</h2>
        <table style="width:100%;border-collapse:collapse;">
        <tr style="border-bottom:2px solid #444;">
          <th style="text-align:left;padding:10px 14px;"></th>
          <th style="text-align:center;padding:10px;color:#64b5f6;">📋 Regular<br><span style="font-size:10px;opacity:.6">No prepay · {rate}%</span></th>
          <th style="text-align:center;padding:10px;color:#81c784;background:rgba(255,255,255,0.03);">💸 Prepayment<br><span style="font-size:10px;opacity:.6">Target {w_target.value}yr · {rate}%</span></th>
          <th style="text-align:center;padding:10px;color:#f48fb1;">🏧 OD + Prepay<br><span style="font-size:10px;opacity:.6">Target {w_target.value}yr · {od_rate}%</span></th>
          <th style="text-align:center;padding:10px;color:#ffcc80;background:rgba(255,255,255,0.03);">🏦 OD Only<br><span style="font-size:10px;opacity:.6">No prepay · {od_rate}%</span></th>
        </tr>
        {R('Tenure',           mf(s1.tenure_months), mf(s2.tenure_months), mf(s3.tenure_months), mf(s4.tenure_months))}
        {R('Monthly EMI',      fmt(s1.emi), fmt(s2.emi), fmt(s3.emi), fmt(s4.emi))}
        {R('Annual Prepayment','—', (fmt(pp2)+'/yr') if pp2 else '—', (fmt(pp3)+'/yr') if pp3 else '—', '—')}
        {R('OD Parked Balance','—','—', fmt(od_bal) if od_bal else '—', fmt(od_bal) if od_bal else '—')}
        {R('Total Prepaid',    '—', fmt(pp2*w_target.value) if pp2 else '—', fmt(pp3*w_target.value) if pp3 else '—', '—')}
        {R('Total Interest',   fmt(s1.total_interest), fmt(s2.total_interest), fmt(s3.total_interest), fmt(s4.total_interest), True)}
        {R('Interest / Loan',
           f'{s1.total_interest/loan*100:.1f}%',
           f'{s2.total_interest/loan*100:.1f}%',
           f'{s3.total_interest/loan*100:.1f}%',
           f'{s4.total_interest/loan*100:.1f}%')}
        {R('Eff. Rate (p.a.)',
           f'{s1.total_interest/loan/(s1.tenure_months/12)*100:.2f}%',
           f'{s2.total_interest/loan/(s2.tenure_months/12)*100:.2f}%',
           f'{s3.total_interest/loan/(s3.tenure_months/12)*100:.2f}%',
           f'{s4.total_interest/loan/(s4.tenure_months/12)*100:.2f}%')}
        {R('Interest Saved',   '—',
           f'<span style="color:#69f0ae;font-weight:700">{fmt(saved2)}</span>',
           f'<span style="color:#f48fb1;font-weight:700">{fmt(saved3)}</span>',
           f'<span style="color:#ffcc80;font-weight:700">{fmt(saved4)}</span>')}
        <tr style="border-bottom:2px solid #444;"><td colspan="5" style="padding:8px 14px;font-size:11px;opacity:.5;letter-spacing:1px;">OPPORTUNITY COST @ {inv_ret}% p.a.</td></tr>
        {R('If Invested Instead', '—', _inv2_html, _inv3_html, _inv4_html)}
        {R('Net Cash Flow',       '—', _net2_html, _net3_html, _net4_html, True)}
        {R('Total Outflow',    fmt(s1.total_payment), fmt(s2.total_payment), fmt(s3.total_payment), fmt(s4.total_payment))}
        </table></div>'''
        display(HTML(tbl))

        # ── Charts ──
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        C = {'1': '#64b5f6', '2': '#81c784', '3': '#f48fb1', '4': '#ffcc80'}

        # -- Balance curves --
        ax = axes[0]
        for s, c, lb in [(s1,C['1'],'Regular'),(s2,C['2'],'Prepay'),(s3,C['3'],'OD+Prepay'),(s4,C['4'],'OD Only')]:
            ax.plot([r.month for r in s.rows], [r.closing_balance for r in s.rows],
                    color=c, lw=2, label=f'{lb} ({s.tenure_months}mo)')
        ax.axvline(target_mo, color='grey', ls=':', alpha=.5)
        ax.set_title('Outstanding Balance', fontsize=13, fontweight='bold')
        ax.set_xlabel('Month'); ax.set_ylabel('₹')
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x/1e5:,.0f}L'))
        ax.legend(fontsize=8)

        # -- Interest bars --
        ax = axes[1]
        vals = [s1.total_interest, s2.total_interest, s3.total_interest, s4.total_interest]
        bars = ax.bar(['Regular','Prepay','OD+Prepay','OD Only'], vals,
                      color=[C['1'],C['2'],C['3'],C['4']], edgecolor='white', width=.5)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*.01,
                    fmt(v), ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_title('Total Interest Paid', fontsize=13, fontweight='bold')
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x/1e5:,.0f}L'))

        # -- Cumulative interest --
        ax = axes[2]
        for s, c, lb in [(s1,C['1'],'Regular'),(s2,C['2'],'Prepay'),(s3,C['3'],'OD+Prepay'),(s4,C['4'],'OD Only')]:
            cum = np.cumsum([r.interest for r in s.rows])
            ax.plot(range(1, len(cum)+1), cum, color=c, lw=2, label=lb)
        ax.axvline(target_mo, color='grey', ls=':', alpha=.5)
        ax.set_title('Cumulative Interest', fontsize=13, fontweight='bold')
        ax.set_xlabel('Month'); ax.set_ylabel('₹')
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x/1e5:,.0f}L'))
        ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()

        # ── Opportunity Cost & Net Cash Flow Chart ──
        fig2, (axL, axR) = plt.subplots(1, 2, figsize=(16, 5))

        # -- Left: stacked breakdown per strategy --
        x_pos = np.arange(2)
        labels_2 = ['Prepayment', 'OD + Prepay']
        w_bar = 0.28

        grp_saved  = [saved2, saved3]
        grp_invest = [corpus_pp2, corpus_pp3 + corpus_od_lump]
        grp_net    = [net_pp2, net_pp3]

        b1 = axL.bar(x_pos - w_bar, grp_saved, w_bar,
                      label='Interest Saved (guaranteed)', color=[C['2'], C['3']], edgecolor='white')
        b2 = axL.bar(x_pos, grp_invest, w_bar,
                      label=f'If Invested @ {inv_ret}%', color=['#a5d6a7', '#f8bbd0'], edgecolor='white')
        net_colors = ['#69f0ae' if v >= 0 else '#ef5350' for v in grp_net]
        b3 = axL.bar(x_pos + w_bar, [abs(v) for v in grp_net], w_bar,
                      label='Net Cash Flow', color=net_colors, edgecolor='white')

        for bars, vals in [(b1, grp_saved), (b2, grp_invest), (b3, grp_net)]:
            for bar, v in zip(bars, vals):
                h = bar.get_height()
                if h > 0:
                    prefix = '+' if vals is grp_net and v >= 0 else ('-' if vals is grp_net and v < 0 else '')
                    axL.text(bar.get_x() + bar.get_width()/2, h + max(max(grp_saved), max(grp_invest)) * 0.01,
                             f'{prefix}{fmt(abs(v))}', ha='center', va='bottom', fontsize=8, fontweight='bold')

        axL.axhline(0, color='#555', lw=0.5)
        axL.set_xticks(x_pos)
        axL.set_xticklabels(labels_2, fontsize=11)
        axL.set_title(f'Opportunity Cost @ {inv_ret}% p.a.', fontsize=13, fontweight='bold')
        axL.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e5:,.0f}L'))
        axL.legend(fontsize=8, loc='upper left')

        # Annotate OD breakdown
        if od_bal > 0 and corpus_pp3 + corpus_od_lump > 0:
            axL.annotate(f'(prepay {fmt(corpus_pp3)} + OD bal {fmt(corpus_od_lump)})',
                         xy=(1, corpus_pp3 + corpus_od_lump), fontsize=7, ha='center',
                         color='#aaa', xytext=(0, 14), textcoords='offset points')

        # -- Right: net cash flow waterfall --
        categories = ['Interest\nSaved', f'Opportunity\nCost @{inv_ret}%', 'NET']
        for idx, (label, clr, s_saved, s_corpus, s_net) in enumerate([
            ('Prepay', C['2'], saved2, corpus_pp2, net_pp2),
            ('OD+Prepay', C['3'], saved3, corpus_pp3 + corpus_od_lump, net_pp3),
        ]):
            axR.barh(2 - idx * 3, s_saved, color=clr, height=0.6, label=f'{label} — Saved' if idx == 0 else '')
            axR.barh(1 - idx * 3, -s_corpus, color=clr, height=0.6, alpha=0.4, label=f'{label} — Opp. Cost' if idx == 0 else '')
            net_clr = '#69f0ae' if s_net >= 0 else '#ef5350'
            axR.barh(0 - idx * 3, s_net, color=net_clr, height=0.6)
            # Labels
            axR.text(s_saved + max(saved2, saved3) * 0.02, 2 - idx * 3, fmt(s_saved), va='center', fontsize=8, fontweight='bold')
            axR.text(-s_corpus - max(saved2, saved3) * 0.02, 1 - idx * 3, f'-{fmt(s_corpus)}', va='center', ha='right', fontsize=8, fontweight='bold')
            sign = '+' if s_net >= 0 else ''
            nx = s_net + (max(saved2, saved3) * 0.02 if s_net >= 0 else -max(saved2, saved3) * 0.02)
            axR.text(nx, 0 - idx * 3, f'{sign}{fmt(s_net)}', va='center', ha='left' if s_net >= 0 else 'right',
                     fontsize=9, fontweight='bold', color=net_clr)

        axR.set_yticks([2, 1, 0, -1, -2, -3])
        axR.set_yticklabels(['Saved', 'Opp.Cost', 'NET\n(Prepay)', 'Saved', 'Opp.Cost', 'NET\n(OD+Prepay)'], fontsize=8)
        axR.axvline(0, color='#555', lw=1)
        axR.set_title('Net Cash Flow Waterfall', fontsize=13, fontweight='bold')
        axR.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e5:,.0f}L'))

        plt.tight_layout()
        plt.show()

        # ── Conclusion ──
        best_lbl = 'Prepayment' if saved2 >= saved3 else 'OD + Prepayment'
        best_saved = max(saved2, saved3)
        other_lbl = 'OD + Prepayment' if saved2 >= saved3 else 'Prepayment'
        other_saved = min(saved2, saved3)
        diff = abs(saved2 - saved3)
        best_clr = C['2'] if saved2 >= saved3 else C['3']

        od_note = ''
        if od_bal > 0:
            od_note = (f'<li>With the OD strategy, your <b>{fmt(od_bal)}</b> parked balance'
                       f' stays <b>100 % liquid</b> — withdraw anytime for emergencies.'
                       f' Prepayments are <b>gone permanently</b>.</li>')

        od_be_note = ''
        if od_bal > 0:
            od_pct = od_bal / loan * 100
            od_be_note = (f'<li>🏦 <b>OD Only:</b> just parking <b>{fmt(od_bal)}</b> '
                          f'(<b>{od_pct:.1f}%</b> of loan) at {od_rate}% — no prepayments — '
                          f'saves <b style="color:#ffcc80">{fmt(saved4)}</b> in interest and '
                          f'closes <b>{s1.tenure_months - s4.tenure_months} months earlier</b> '
                          f'({mf(s4.tenure_months)}). Money stays 100% liquid.</li>')

        pp2_note = ''
        if pp2 > 0:
            mult2 = saved2 / (pp2 * w_target.value)
            pp2_note = (f'<li>Every ₹1 prepaid in Strategy 2 saves'
                        f' <b>₹{mult2:.2f}</b> in interest'
                        f' — a <b>{mult2*100:.0f}%</b> return.</li>')

        invest_note = ''
        if pp2 > 0 and inv_ret > 0:
            if net_pp2 >= 0:
                invest_note = (f'<li>💰 <b>Prepay net cash flow: <span style="color:#69f0ae">+{fmt(net_pp2)}</span></b> — '
                               f'interest saved ({fmt(saved2)}) exceeds what investing at {inv_ret}% would yield ({fmt(corpus_pp2)}). '
                               f'<b>Prepaying wins</b>, and it\'s guaranteed &amp; tax-free.</li>')
            else:
                invest_note = (f'<li>💰 <b>Prepay net cash flow: <span style="color:#ef5350">{fmt(net_pp2)}</span></b> — '
                               f'investing {fmt(pp2)}/yr at {inv_ret}% would grow to <b>{fmt(corpus_pp2)}</b>, '
                               f'which is <b>{fmt(abs(net_pp2))} more</b> than the {fmt(saved2)} interest saved. '
                               f'Markets win on paper, but returns are <em>not guaranteed</em>.</li>')

        od_invest_note = ''
        if od_bal > 0 and inv_ret > 0:
            if net_pp3 >= 0:
                od_invest_note = (f'<li>💰 <b>OD+Prepay net cash flow: <span style="color:#69f0ae">+{fmt(net_pp3)}</span></b> — '
                                  f'interest saved ({fmt(saved3)}) beats the opportunity cost '
                                  f'(prepay growth {fmt(corpus_pp3)} + OD growth {fmt(od_missed_growth)}). '
                                  f'OD strategy wins <em>and</em> keeps {fmt(od_bal)} liquid.</li>')
            else:
                od_invest_note = (f'<li>💰 <b>OD+Prepay net cash flow: <span style="color:#ef5350">{fmt(net_pp3)}</span></b> — '
                                  f'investing the same capital at {inv_ret}% beats the {fmt(saved3)} interest saved '
                                  f'by <b>{fmt(abs(net_pp3))}</b>. However, OD keeps {fmt(od_bal)} fully liquid.</li>')

        conc = f'''
        <div style="background:#263238;border-radius:12px;padding:22px;color:#ccc;margin-top:12px;
                    font-family:-apple-system,BlinkMacSystemFont,sans-serif;
                    border-left:4px solid {best_clr};font-size:14px;">
          <h3 style="margin:0 0 10px;color:#fff;">📌 Conclusion</h3>
          <ul style="line-height:2.1;margin:0;">
            <li><b>Regular loan</b> ({rate}%, {s1.tenure_months//12}yr) costs
                <b>{fmt(s1.total_interest)}</b> in interest.</li>
            <li><b>Prepayment</b> ({fmt(pp2)}/yr × {w_target.value}yr) closes in
                <b>{mf(s2.tenure_months)}</b>, saving <b style="color:#81c784">{fmt(saved2)}</b>.</li>
            <li><b>OD + Prepayment</b> ({fmt(pp3)}/yr + {fmt(od_bal)} parked, {od_rate}%)
                closes in <b>{mf(s3.tenure_months)}</b>, saving <b style="color:#f48fb1">{fmt(saved3)}</b>.</li>
            {pp2_note}
            {invest_note}
            {od_invest_note}
            {od_note}
            {od_be_note}
            <li style="font-size:15px;margin-top:6px;">
              <b>Best strategy → <span style="color:{best_clr}">{best_lbl}</span></b>
              — saves <b>{fmt(best_saved)}</b> total
              ({fmt(diff)} more than {other_lbl}).</li>
          </ul>
        </div>'''
        display(HTML(conc))

# ── Wire up & display ────────────────────────────────────
for w in [w_loan, w_rate, w_tenure, w_target, w_od_rate, w_od_bal, w_invest]:
    w.observe(refresh, names='value')

left  = widgets.VBox([widgets.HTML('<b>📋 Loan</b>'), w_loan, w_rate, w_tenure])
mid   = widgets.VBox([widgets.HTML('<b>🎯 Strategy</b>'), w_target, w_od_rate, w_od_bal])
right = widgets.VBox([widgets.HTML('<b>📊 Analysis</b>'), w_invest])
panel = widgets.HBox([left, mid, right],
          layout=widgets.Layout(border='1px solid #ddd', border_radius='10px',
                                padding='12px', margin='0 0 12px 0'))
display(panel, out)
refresh()

Output()